# HW1 Time-Series Data Analysis: Exploring EHR Data (PhysioNet 2012)

Place `P12_preprocess.py`, `Outcomes-a.txt`, `Outcomes-b.txt`, and both data archives or folders beside this notebook. Use `/content` on Colab.

Run the installation cell once and restart the runtime before running the remaining cells.

In [ ]:
%pip install numpy==2.5.4 pandas==3.1.1
%pip install scikit-learn==1.9.1 lightgbm==4.7.0
%pip install matplotlib==3.11.2

In [ ]:
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections.abc import Callable

import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

In [ ]:
import P12_preprocess as pp
from P12_preprocess import FEATURE_COLS

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

# Extract data if needed
for d in (pp.SET_A_DIR, pp.SET_B_DIR):
    if not d.is_dir():
        with tarfile.open(f"{d}.tar.gz") as tar:
            tar.extractall(d.parent, filter="data")

## P1(a) Data structure

In [ ]:
PID = 132539  # Example for P1(a) to P1(e)
raw = pd.read_csv(pp.SET_A_DIR / f"{PID}.txt")
print("shape:", raw.shape)
raw.head(10)

In row 7 (`00:07, HR, 73`), `Time` means 7 minutes after ICU admission, `Parameter` is heart rate,
and `Value` is 73 bpm.

We scanned all Set A files. General descriptors are recorded at admission, while time series
variables can be measured repeatedly. `Weight` belongs to both groups.

In [ ]:
long_a = pd.concat(
    [pd.read_csv(f).assign(pid=int(f.stem)) for f in sorted(pp.SET_A_DIR.glob("*.txt"))],
    ignore_index=True,
)
n_patients = long_a.pid.nunique()

per_patient = long_a.groupby(["Parameter", "pid"]).size()
summary = pd.DataFrame({
    "n_patients": per_patient.groupby(level=0).size(),
    "max_per_patient": per_patient.groupby(level=0).max(),
    "frac_at_00:00": long_a.Time.eq("00:00").groupby(long_a.Parameter).mean().round(4),
})
summary.sort_values("frac_at_00:00", ascending=False)

In [ ]:
is_descriptor = (
    summary.n_patients.eq(n_patients) & summary.max_per_patient.eq(1) & summary["frac_at_00:00"].eq(1)
)
print(f"Once-at-admission ({is_descriptor.sum()}):", summary.index[is_descriptor].tolist())
print(f"Repeated ({(~is_descriptor).sum()}):", summary.index[~is_descriptor].tolist())

# Weight is recorded at admission and later
w = long_a[long_a.Parameter == "Weight"]
print(f"\nWeight: at 00:00 for {w[w.Time == '00:00'].pid.nunique()}/{n_patients} patients, "
      f"{(w.Time != '00:00').sum()} later measurements")

General descriptors (6): `RecordID`, `Age`, `Gender`, `Height`, `ICUType`, and `Weight`.

Time series variables (37): the repeated variables listed above, including `Weight`.

## P1(b) Relative time conversion

Source: `set-a/132539.txt`. 

Output: `132539_hours.txt`.

We convert `Time` using `HH + MM / 60`. All rows stay in order, and `Parameter` and `Value` are unchanged.

In [ ]:
SRC = pp.SET_A_DIR / f"{PID}.txt"
DST = Path(f"{PID}_hours.txt")

df_b = pd.read_csv(SRC, dtype=str)  # Keep original text
hh, mm = df_b.Time.str.split(":", expand=True).astype(int).T.to_numpy()
df_b["Time"] = hh + mm / 60
df_b.to_csv(DST, index=False)

# Check row order and unchanged values
src, out = pd.read_csv(SRC, dtype=str), pd.read_csv(DST, dtype=str)
assert len(src) == len(out) and src[["Parameter", "Value"]].equals(out[["Parameter", "Value"]])
print("rows:", len(out))
pd.concat({"source": src, "converted": out}, axis=1).head(10)

## P1(c) Time series structure

In [ ]:
pivot_a = pp.process_set(pp.SET_A_DIR, pp.OUTCOMES_A)
pivot_b = pp.process_set(pp.SET_B_DIR, pp.OUTCOMES_B)
for name, p in [("Set A", pivot_a), ("Set B", pivot_b)]:
    print(f"{name}: {p.shape}, {p.pid.nunique()} patients")

In [ ]:
print("Raw (long format):", raw.shape)
display(raw.head(15))

one = pivot_a[pivot_a.pid == PID]
print("Processed (one row per 2-h bin):", one.shape)
display(one)

In [ ]:
STATIC = ["Age", "Gender", "Height", "ICUType", "Weight"]
print("Distinct values per static column:", one[STATIC].nunique(dropna=False).to_dict())
print("Timestamps:", one.timestamp.tolist())

Missing values are `NaN`. Bins with no records are absent.

The first nonmissing bin value of each static descriptor is copied to every row.
Later `Weight` values are overwritten.

`target` is `In-hospital_death` (0: survived, 1: died).
`seg_entry_cnt` counts distinct recorded times in each bin.
Bins are 2 hours wide and include their end time. Admission records go in bin 2.

## P1(d) Window selection

In [ ]:
def window(df: pd.DataFrame, start: int = 12, end: int = 36) -> pd.DataFrame:
    """
    Keep timestamps from start to end, inclusive
    """
    return df[df.timestamp.between(start, end)].reset_index(drop=True)


win_a = window(pivot_a)
win_b = window(pivot_b)

print("Before:", one.shape)
display(one)

after = win_a[win_a.pid == PID]
print("After:", after.shape, "timestamps:", after.timestamp.tolist())
display(after)

We predict at hour 36 to use enough observations while leaving time for care decisions.
Starting at hour 12 aims to reduce the effect of admission procedures.

We keep bin labels from 12 to 36. Bin 12 covers `(10, 12]`, so the raw time range is `(10, 36]`.

## P1(e) Missing value imputation

We treat `Height`, `Weight`, and `Gender` values of `-1` as missing.
Within each patient, we use `ffill()` then `bfill()` to fill gaps with nearby measurements.
All measurements are available by hour 36.

Remaining gaps use Set A medians for continuous variables and modes for `Gender` and `ICUType`.
Medians reduce the effect of outliers. For `MechVent`, we assume 0 if the whole window is missing.

In [ ]:
SENTINEL_COLS = ["Weight", "Gender"]
CATEGORICAL = ["Gender", "ICUType"]

def clean_sentinels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Treat -1 as missing
    """
    df = df.copy()
    df[SENTINEL_COLS] = df[SENTINEL_COLS].replace(-1, np.nan)
    return df

def fit_fallback(win: pd.DataFrame) -> pd.Series:
    """
    Fit median and mode values on training data
    """
    clean = clean_sentinels(win)
    fallback = clean[FEATURE_COLS].median()
    fallback[CATEGORICAL] = clean[CATEGORICAL].mode().iloc[0]
    fallback["MechVent"] = 0
    return fallback

def impute(df: pd.DataFrame, fallback: pd.Series) -> pd.DataFrame:
    """
    Fill within patients, then use training values
    """
    df = clean_sentinels(df)
    filled = df.groupby("pid")[FEATURE_COLS].ffill().groupby(df.pid).bfill()
    df[FEATURE_COLS] = filled.fillna(fallback)
    return df

fallback = fit_fallback(win_a) # Training data only
imp_a = impute(win_a, fallback)
imp_b = impute(win_b, fallback)
print("Remaining NaNs - Set A:", imp_a[FEATURE_COLS].isna().sum().sum(), "| Set B:", imp_b[FEATURE_COLS].isna().sum().sum())

In [ ]:
before = win_a[win_a.pid == PID]
print(f"Patient {PID} - before: {before[FEATURE_COLS].isna().sum().sum()} NaNs")
display(before)

after = imp_a[imp_a.pid == PID]
print(f"After: {after[FEATURE_COLS].isna().sum().sum()} NaNs")
display(after)

## P1(f) Zero padding

Patients with no rows in the window are excluded. Others use 13 time points from hour 12 to 36.
Missing bins get zero feature rows and `seg_entry_cnt = 0`.

In [ ]:
STEPS = np.arange(12, 37, 2)  # 13 time points

def zero_pad(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fill missing bins with zero feature rows
    """
    grid = pd.MultiIndex.from_product([df.pid.unique(), STEPS], names=["pid", "timestamp"])
    out = df.set_index(["pid", "timestamp"]).reindex(grid)
    out["target"] = out.groupby(level="pid").target.transform("first")
    return out.fillna(0).astype({"target": int, "seg_entry_cnt": int}).reset_index()

for name, piv, imp in [("Set A", pivot_a, imp_a), ("Set B", pivot_b, imp_b)]:
    excluded = sorted(set(piv.pid) - set(imp.pid))
    print(f"{name}: excluded {len(excluded)} patients with 0 rows in window: {excluded}")

pad_a, pad_b = zero_pad(imp_a), zero_pad(imp_b)
X_train = pad_a[FEATURE_COLS].to_numpy().reshape(-1, len(STEPS), len(FEATURE_COLS))
X_test = pad_b[FEATURE_COLS].to_numpy().reshape(-1, len(STEPS), len(FEATURE_COLS))
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

In [ ]:
PID_PAD = 132772  # 7 observed bins
before = imp_a[imp_a.pid == PID_PAD]
print(f"Patient {PID_PAD} - before padding:", before.shape)
display(before)

after = pad_a[pad_a.pid == PID_PAD]
print("After padding:", after.shape)
display(after)

## P1(g) Labels and class distribution

In [ ]:
# Match one label to each patient in X
y_train = pad_a.groupby("pid", sort=False).target.first().to_numpy()
y_test = pad_b.groupby("pid", sort=False).target.first().to_numpy()
assert len(y_train) == len(X_train) and len(y_test) == len(X_test)

def class_dist(y: np.ndarray) -> pd.DataFrame:
    counts = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({"count": counts, "proportion": (counts / counts.sum()).round(4)})

pd.concat(
    {"y_train (Set A)": class_dist(y_train), "y_test (Set B)": class_dist(y_test)}, axis=1
).rename_axis("In-hospital_death")

## P2 Baseline training

We keep the same patients and window as P1, but skip imputation and padding.
Each patient has 185 features from five bin statistics (`last`, `mean`, `min`, `max`, `count`) and 5 static descriptors.

We use LightGBM because it handles missing values and tabular data.
To help reduce overfitting, we use a small learning rate, limited leaves, and subsampling. We compare P1 inputs and summary features using five stratified folds on Set A. P1 imputation is fitted within each training fold. Set B is used only for evaluation.

Run all cells in order with `SEED = 42`. AUPRC uses average precision.
All tested seeds are listed below.

In [ ]:
SEED = 42
print("lightgbm", lgb.__version__)

In [ ]:
DYNAMIC = [c for c in FEATURE_COLS if c not in STATIC]
STATS = ["last", "mean", "min", "max", "count"]

def summary_features(win: pd.DataFrame, pids: np.ndarray) -> pd.DataFrame:
    """
    Summarize each patient's bins
    """
    g = clean_sentinels(win).groupby("pid")
    parts = [g[DYNAMIC].agg(s).add_suffix(f"_{s}") for s in STATS] + [g[STATIC].first()]
    return pd.concat(parts, axis=1).loc[pids]

# Keep P1 patient order
F_train = summary_features(win_a, pad_a.pid.unique())
F_test = summary_features(win_b, pad_b.pid.unique())
print("F_train:", F_train.shape, "| F_test:", F_test.shape)

In [ ]:
PARAMS = dict(
    n_estimators=500,
    learning_rate=0.02,    # Small boosting steps
    num_leaves=15,
    min_child_samples=20,  # Limit tree size
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    reg_lambda=1.0,        # Reduce overfitting
    deterministic=True,
    force_col_wise=True,
    verbose=-1,            # Reproducible training
)

def make_model(seed: int = SEED) -> lgb.LGBMClassifier:
    return lgb.LGBMClassifier(**PARAMS, random_state=seed)

def scores(y: np.ndarray, p: np.ndarray) -> dict[str, float]:
    return {"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p)}

def cv_scores(
    build: Callable[[np.ndarray, np.ndarray], tuple[pd.DataFrame, pd.DataFrame]],
    y: np.ndarray,
    seed: int = SEED,
    k: int = 5,
) -> dict[str, float]:
    """
    Validate on Set A; fit preprocessing in each training fold
    """
    oof = np.zeros(len(y))
    for tr, va in StratifiedKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        X_tr, X_va = build(tr, va)
        oof[va] = make_model(seed).fit(X_tr, y[tr]).predict_proba(X_va)[:, 1]

    return scores(y, oof)

In [ ]:
pids_train = pad_a.pid.unique() # Label order

def p1_tensor(win: pd.DataFrame, pids: np.ndarray, fallback: pd.Series) -> pd.DataFrame:
    """
    Impute, pad, and flatten each patient
    """
    padded = zero_pad(impute(win[win.pid.isin(pids)], fallback))
    assert (padded.pid.unique() == pids).all()
    return pd.DataFrame(padded[FEATURE_COLS].to_numpy().reshape(len(pids), -1))

def p1_split(tr: np.ndarray, va: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    fallback_tr = fit_fallback(win_a[win_a.pid.isin(pids_train[tr])])  # Training fold only
    return p1_tensor(win_a, pids_train[tr], fallback_tr), p1_tensor(win_a, pids_train[va], fallback_tr)

def summary_split(tr: np.ndarray, va: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    return F_train.iloc[tr], F_train.iloc[va]  # No statistics shared across patients

# Compare feature sets on Set A
candidates = {"P1 tensor as-is (13x41 flattened)": p1_split, "Summary features (final)": summary_split}
pd.DataFrame({name: cv_scores(build, y_train) for name, build in candidates.items()}).T.round(4)

In [ ]:
# Fit on Set A and evaluate on Set B
model = make_model().fit(F_train, y_train)
test = scores(y_test, model.predict_proba(F_test)[:, 1])

TIERS = [("Strong", 0.83, 0.46), ("Medium", 0.80, 0.43), ("Simple", 0.75, 0.38)]
tier = next((name for name, roc, prc in TIERS if test["AUROC"] >= roc and test["AUPRC"] >= prc), "none")
print(f"Set B (seed {SEED}): AUROC {test['AUROC']:.4f} | AUPRC {test['AUPRC']:.4f} -> tier: {tier}")

In [ ]:
# All tested seeds; final seed is 42
SEEDS = [42, 7, 123, 256, 1001]
runs = pd.DataFrame(
    {s: scores(y_test, make_model(s).fit(F_train, y_train).predict_proba(F_test)[:, 1]) for s in SEEDS}
).T.rename_axis("seed")
pd.concat([runs, runs.agg(["mean", "std"])]).round(4)

## P3 Feature resolution

How much numerical detail does mortality prediction need?

We compare original P2 features with 2 to 8 quantile levels using five folds on Set A.
We use five seeds for splitting and training. Settings share folds within each seed.

Boundaries use training folds only. Gender, ICUType, counts, and missing values stay unchanged.
LightGBM's internal binning is unchanged.

In [ ]:
P3_LEVELS = [2, 3, 4, 5, 6, 7, 8]
P3_SEEDS = [42, 7, 123, 256, 1001]

def fit_quantile_edges(X: pd.DataFrame, levels: int) -> dict[str, np.ndarray]:
    """
    Fit numeric boundaries on training data
    """
    if levels < 2:
        raise ValueError("levels must be at least 2")

    quantiles = np.arange(1, levels) / levels
    edges = {}
    for col in X.columns:
        if col in CATEGORICAL or col.endswith("_count"):
            continue
        values = X[col].dropna()
        if values.nunique() <= levels:
            edges[col] = np.sort(values.unique())[1:]
        else:
            edges[col] = np.unique(values.quantile(quantiles).to_numpy())
    return edges

def apply_quantile_edges(X: pd.DataFrame, edges: dict[str, np.ndarray]) -> pd.DataFrame:
    """
    Apply fitted boundaries and preserve missing values
    """
    out = X.copy()
    for col, cuts in edges.items():
        codes = pd.Series(np.searchsorted(cuts, X[col], side="right"), index=X.index)
        out[col] = codes.where(X[col].notna())
    return out

In [ ]:
p3_settings = [("Original", None)] + [(f"{n} levels", n) for n in P3_LEVELS]
p3_oof = {}
p3_run_rows = []
p3_fold_rows = []

for p3_seed in P3_SEEDS:
    p3_seed_oof = {name: np.zeros(len(y_train)) for name, _ in p3_settings}
    p3_cv = StratifiedKFold(5, shuffle=True, random_state=p3_seed)

    for p3_fold, (tr, va) in enumerate(p3_cv.split(F_train, y_train), start=1):
        for p3_name, p3_levels in p3_settings:
            p3_train = F_train.iloc[tr]
            p3_valid = F_train.iloc[va]

            if p3_levels is not None:
                p3_edges = fit_quantile_edges(p3_train, p3_levels)
                p3_train = apply_quantile_edges(p3_train, p3_edges)
                p3_valid = apply_quantile_edges(p3_valid, p3_edges)

            p3_model = make_model(p3_seed).fit(p3_train, y_train[tr])
            p3_prob = p3_model.predict_proba(p3_valid)[:, 1]
            p3_seed_oof[p3_name][va] = p3_prob
            p3_fold_rows.append({
                "seed": p3_seed, "fold": p3_fold, "Input": p3_name,
                **scores(y_train[va], p3_prob),
            })
        print(f"Seed {p3_seed}, fold {p3_fold}/5 complete")

    p3_oof[p3_seed] = p3_seed_oof
    p3_baseline = scores(y_train, p3_seed_oof["Original"])
    for p3_name, p3_prob in p3_seed_oof.items():
        p3_score = scores(y_train, p3_prob)
        p3_run_rows.append({
            "seed": p3_seed, "Input": p3_name, **p3_score,
            "Delta AUROC": p3_score["AUROC"] - p3_baseline["AUROC"],
            "Delta AUPRC": p3_score["AUPRC"] - p3_baseline["AUPRC"],
        })

p3_fold_scores = pd.DataFrame(p3_fold_rows)
p3_runs = pd.DataFrame(p3_run_rows).set_index(["seed", "Input"])
p3_results = p3_runs.groupby(level="Input", sort=False).agg(["mean", "std"])
p3_runs.to_csv("p3_resolution_runs.csv")
p3_results.to_csv("p3_resolution_summary.csv")
print("Set A OOF mean and standard deviation across seeds")
p3_results.round(4)

In [ ]:
p3_order = [f"{n} levels" for n in P3_LEVELS] + ["Original"]
p3_positions = np.arange(len(p3_order))
p3_fig, p3_axes = plt.subplots(1, 2, figsize=(10, 3.5))
for p3_ax, p3_metric in zip(p3_axes, ["AUROC", "AUPRC"]):
    p3_values = p3_results.loc[p3_order, p3_metric]
    p3_ax.errorbar(
        p3_positions, p3_values["mean"], yerr=p3_values["std"],
        marker="o", capsize=3,
    )
    p3_ax.set_xticks(p3_positions, [str(n) for n in P3_LEVELS] + ["Original"])
    p3_ax.set_title(p3_metric)
    p3_ax.set_xlabel("Numeric levels")
    p3_ax.set_ylabel("Set A OOF mean and SD")
    p3_ax.grid(axis="y", alpha=0.3)
p3_fig.tight_layout()
p3_fig.savefig("p3_feature_resolution.png", dpi=160)
plt.show()

Two levels reduced mean AUPRC from 0.4698 to 0.4035.
Seven and eight levels reached 0.4648 and 0.4658.

Eight levels performed similarly to the original inputs in this Set A experiment.
This may partly reflect LightGBM's internal binning, which already groups numerical values.